# 02 · Contrastive fine-tuning  *(stage 5b)*Fine-tunes `bge-small-en-v1.5` on 2,000 maintainer-marked duplicate pairs.**~5 minutes on a T4.**Result: recall@10 **0.3016 → 0.3214**.### Upload| file | produced by ||---|---|| `pairs_train.jsonl.gz` | `python -m src.finetune --export-pairs colab/pairs_train.jsonl.gz` || `texts_clean.jsonl.gz` | `python -m src.embed --export-texts ... --clean` |### Why MultipleNegativesRankingLossMaintainers only ever give us **positive** pairs — "this duplicates that". MNRLneeds nothing else: for each pair it treats every *other* pair in the batch as anegative. Bigger batches mean more negatives and a harder task.### Leakage guardsTraining uses the **train split only** (2,012 pairs); the 504 test pairs arenever seen. The split is **chronological**, so the model cannot learn fromduplicates filed after its own test queries.

In [ ]:
!pip install -q sentence-transformers huggingface_hub

In [ ]:
import torch, gcgc.collect(); torch.cuda.empty_cache()assert torch.cuda.is_available(), "Runtime > Change runtime type > T4 GPU"print(torch.cuda.get_device_name(0),      f"{torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

In [ ]:
import gzip, json, random, numpy as np, torchfrom sentence_transformers import (InputExample, SentenceTransformer, losses)from sentence_transformers.evaluation import EmbeddingSimilarityEvaluatorfrom torch.utils.data import DataLoader# Seeded so a re-run reproduces. GPU training is not bit-exact even so -- expect# small drift, and re-measure rather than reusing the previous number.random.seed(0); np.random.seed(0); torch.manual_seed(0); torch.cuda.manual_seed_all(0)rows = [json.loads(l) for l in gzip.open("pairs_train.jsonl.gz", "rt")]random.Random(0).shuffle(rows)val, train = rows[:150], rows[150:]print(f"{len(train)} train / {len(val)} holdout")

The held-out 150 pairs are a smoke signal, not a metric. 2,000 pairs is easy tomemorise; if this score climbs and then falls between epochs, drop to 1–2 epochs.The real evaluation is recall@10 on the 504-pair test split, back on the laptop.

In [ ]:
model = SentenceTransformer("BAAI/bge-small-en-v1.5")loader = DataLoader([InputExample(texts=[r["a"], r["p"]]) for r in train],                    shuffle=True, batch_size=32, drop_last=True)loss = losses.MultipleNegativesRankingLoss(model)ev = EmbeddingSimilarityEvaluator([r["a"] for r in val], [r["p"] for r in val],                                  [1.0] * len(val), name="heldout")model.fit(train_objectives=[(loader, loss)], evaluator=ev, epochs=3,          warmup_steps=int(len(loader) * 3 * 0.1),          optimizer_params={"lr": 2e-5},          output_path="ft-model", show_progress_bar=True)print("peak GPU:", torch.cuda.max_memory_allocated() / 1e9, "GB")

### Push to the Hub`create_repo(exist_ok=True)` then `upload_folder` — **not** `push_to_hub`.`push_to_hub` tries to create the repo and returns **409 Conflict** if it alreadyexists.Your HF username is probably **not** your GitHub username. Check with`huggingface_hub.whoami()`; a wrong namespace gives **403 Forbidden**. The tokenneeds **write** scope.

In [ ]:
from huggingface_hub import login, HfApilogin()REPO = "Musab6969/bge-small-vscode-dup"api = HfApi()api.create_repo(REPO, repo_type="model", private=False, exist_ok=True)api.upload_folder(folder_path="ft-model", repo_id=REPO, repo_type="model",                  commit_message="bge-small fine-tuned on 2,000 vscode duplicate pairs")print("pushed", REPO)

### Re-encode with the fine-tuned model**Keep using the same `model` object.** Constructing a fresh`SentenceTransformer("BAAI/bge-small-en-v1.5")` here would silently encode thecorpus with the *untrained* model — the vectors would load fine and retrievalwould just be worse, with nothing to indicate why.

In [ ]:
import gcgc.collect(); torch.cuda.empty_cache()   # free optimizer state before encodingfrom pathlib import Pathrecs = [json.loads(l) for l in gzip.open("texts_clean.jsonl.gz", "rt")]Path("shards_ft").mkdir(exist_ok=True)for s in range(0, len(recs), 10_000):    chunk = recs[s:s + 10_000]    v = model.encode([r["t"] for r in chunk], batch_size=64,                     normalize_embeddings=True, show_progress_bar=True,                     convert_to_numpy=True).astype(np.float32)    i = s // 10_000    np.save(f"shards_ft/emb_{i:05d}.npy", v)    np.save(f"shards_ft/ids_{i:05d}.npy", np.array([r["n"] for r in chunk], dtype=np.int64))    print("shard", i)!zip -qr shards_ft.zip shards_ftfrom google.colab import filesfiles.download("shards_ft.zip")

### Back on the laptop```bashunzip shards_ft.zippython -m src.embed --import-vectors shards_ft/ --model-key bge-small-ft-duppython -m src.retrieve          # the ablation table```